# MQL-0010 with a FINE-TUNED model (same notebook, same package, fine-tuned weights)

This is the teammate's MQL-0010 notebook with **one difference: the model weights**. A LoRA adapter trained on
`approved_v2_1_mql0010_view` (MQL-0010's registered retrieval view: the 18 holdout databases are not in it) is merged
into the same pinned `Qwen/Qwen2.5-Coder-3B-Instruct` revision, and the unchanged MQL-0010 generator loads the merged
weights. Prompts, retrieval, retries, batching, decoding, checkpoints, `verify_exports` and the export format are all
the original code, so the 9 arms (A0 ... FULL) come out in exactly the same format as the base-model run and are scored
the same way. Changes vs the original: config (adapter + output names, all 8 passes in one session), a new
"fine-tuned weights" cell, the generator's `local_dir`, an adapter check on resume, and the adapter identity in the
status file.

**Setup:** attach (1) the MQL-0010 package dataset (the same unmodified bundle as before) and (2) the training
notebook's **Notebook Output** with the `_q10` adapter. GPU T4 x2, Internet on. Save & Run All (Commit).
`FAST_A0_ONLY = True` (default): only the A0 arm, the fine-tuned model on the zero-shot prompts, ~2 h; the full export
says "partial" (expected) and the last cell writes `predictions/A0/predictions.jsonl` into the zip.
`FAST_A0_ONLY = False`: all 9 arms, ~10 h.
Output: `/kaggle/working/mql0010_ft_predictions.zip`. Arms are named as in the original (A0, D0, ...): when scoring,
label them as the fine-tuned run (e.g. FT-A0); `FT_ADAPTER.json` says which adapter produced them.

---
# MQL-0010: controlled recovery, diversified selection and join-key notes (generate only)

Registration **A4** (pinned by `registration/registration_index.json`), 716 held-out Spider-train questions over 18
databases that are **excluded from retrieval** (view of 2,912 corpus records, hash-checked here).

| pass | generates | batch |
|---|---|---|
| `first:A0`, `first:D0`, `first:E` | zero-shot / MQL-0008 hybrid / diversified-selection prompts, every item | 8 |
| `delta` | EP (E prompt) and EF (E + join-key notes) for the items whose prompt changes | 1 |
| `retry:A0V`, `retry:D0V`, `retry:D0G`, `retry:FULL` | one retry for validator-flagged answers (D0G = generic message) | 1 |

The 9 arms (A0, A0V, D0, D0V, D0G, E, EP, EF, FULL) are assembled from these passes at export and checked by
`scoring.verify_exports`.

- **Generate only.** No MongoDB, no Atlas, no secrets, no external model API, no gold labels in the package.
- **Settings:** Accelerator *GPU T4 x2* (one card used), Internet **on** (public model download at the pinned revision),
  Persistence *Files only*. Attach the package dataset. Attach **no** secrets.
- **Sessions:** session 1 = first passes + delta (~7.6 T4-h), session 2 = retries (~2.4 T4-h). From session 2 on, attach
  the previous version's output and set `RESUME_FROM = "auto"`. Pin the environment after session 1.
- bfloat16 on a **Tesla T4** (emulated, as MQL-0006..0009). Any other GPU stops the run.
- Every plan and prompt is **re-derived here** with the pinned tokenizer and compared with the frozen copy.

In [ ]:
# ---- configuration (edit here only) --------------------------------------------------------------
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SESSION = 1                       # 1 = first passes + delta; 2 = retries (configs/experiment.json -> session_plan)
FAST_A0_ONLY = True               # True = only the A0 arm (fine-tuned zero-shot), ~2 h. False = all 9 arms, ~10 h
RUN_PASSES = ["first:A0"] if FAST_A0_ONLY else ["first:A0", "first:D0", "first:E", "delta",
                                                "retry:A0V", "retry:D0V", "retry:D0G", "retry:FULL"]
RESUME_FROM = None                # None (session 1) | "auto" (attach the previous version's output) | a path
INPUT_ROOT = "/kaggle/input"
PKG_DIR = None                    # None = search INPUT_ROOT for the nl2mql_mql0010_holdout package
OUT_DIR = "/kaggle/working/mql0010_ft_outputs"
OUTPUT_ZIP = "/kaggle/working/mql0010_ft_predictions.zip"
MODEL_CACHE = "/kaggle/working/hf_model"
TIME_BUDGET_HOURS = 11.0          # stop starting new batches after this; checkpoints are kept and exported
MIN_FREE_DISK_GIB = 10.0
MIN_RAM_GIB = 8.0
INSTALL_MISSING = False           # pip-install ONLY missing packages (never upgrades preinstalled ones)

# ---- fine-tuned model ----
ADAPTER_NAME = None               # adapter folder name (e.g. "q3b_r16_ep2_final_q10"); None = the only adapter attached
ADAPTER_DIR = None                # or an explicit path to the adapter folder
MERGED_DIR = "/tmp/mql0010_ft_merged"   # merged weights (~6 GB); /tmp so they aren't saved as notebook output
MAX_SAFE_TRAIN_ROWS = 2901        # approved_v2_1_mql0010_view sft_full; an adapter trained on more rows saw holdout data

# ---- test hooks (leave the defaults on Kaggle) ----
GENERATOR_FACTORY = None          # callable(cfg) -> (run_batch, info, is_oom, cleanup); None = real Qwen on the GPU
FAKE_GPU = None                   # dict replacing the GPU probe (tests only)
COUNT_TOKENS = None               # callable(str) -> int replacing the tokenizer (tests only)
STOP_AFTER_BATCHES = None         # kill-and-resume tests only

In [ ]:
# ---- package integrity: manifest hashes, registration A4 pins, corpus + retrieval view, evaluation inputs ----------
import json, platform, shutil, sys, time
from importlib import metadata
from pathlib import Path

def _find_pkg():
    if PKG_DIR:
        return Path(PKG_DIR)
    hits = []
    for m in Path(INPUT_ROOT).rglob("package_manifest.json"):
        try:
            if json.loads(m.read_text(encoding="utf-8")).get("package") == "nl2mql_mql0010_holdout":
                hits.append(m.parent)
        except (OSError, ValueError):
            pass
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one nl2mql_mql0010_holdout package under {INPUT_ROOT}, found {hits}")
    return hits[0]

PKG = _find_pkg()
sys.path.insert(0, str(PKG / "src"))
import mql0010_core as C

P = C.load_package(PKG)
CFG, M = P["cfg"], P["cfg"]["model"]
if RUN_PASSES is None:
    RUN_PASSES = CFG["session_plan"][str(SESSION)]["passes"]
C.need(RUN_PASSES and set(RUN_PASSES) <= set(C.PASSES) and len(set(RUN_PASSES)) == len(RUN_PASSES), f"RUN_PASSES must be a subset of {C.PASSES}")
RUN_PASSES = [p for p in C.PASSES if p in RUN_PASSES]           # canonical dependency order
print(f"package OK: {PKG}\n  {len(P['manifest']['files'])} files hash-verified; registration index {CFG['registration_index_sha256'][:12]}; "
      f"retrieval view {len(P['view_rows'])} records (18 holdout DBs excluded); evaluation {len(P['ids'])} items; passes: {RUN_PASSES}")

In [ ]:
# ---- environment: dependencies, credentials, GPU + precision, disk, RAM --------------------------------------------
def _ver(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

NEEDED = ["torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]
missing = [p for p in NEEDED if _ver(p) is None]
if missing and GENERATOR_FACTORY is None:
    if not INSTALL_MISSING:
        raise RuntimeError(f"missing packages {missing}: set INSTALL_MISSING = True (installs only the missing ones)")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

attached = [str(p) for p in Path(INPUT_ROOT).rglob("*") if p.name.startswith(".env") or p.suffix in (".pem", ".key")] \
    if Path(INPUT_ROOT).exists() else []
C.need(not attached, f"refusing to run: credential-like files attached as input: {attached[:5]}")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
ENV = {"python": platform.python_version(), "platform": platform.platform(), "packages": {p: _ver(p) for p in NEEDED}}
try:
    _ram = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2)
except (AttributeError, ValueError, OSError):
    _ram = None
HOST = {"ram_gib": _ram, "free_disk_gib": round(shutil.disk_usage(OUT_DIR).free / 2**30, 2)}
GPU = FAKE_GPU if FAKE_GPU is not None else C.K9.gpu_runtime_info()
PRECISION = C.K9.check_precision(CFG, GPU)       # bf16 on the registered T4 (emulated), or STOP
C.need(HOST["free_disk_gib"] >= MIN_FREE_DISK_GIB or GENERATOR_FACTORY is not None, f"free disk {HOST['free_disk_gib']} GiB")
C.need(HOST["ram_gib"] is None or HOST["ram_gib"] >= MIN_RAM_GIB, f"RAM {HOST['ram_gib']} GiB < {MIN_RAM_GIB}")
ENV["gpu"], ENV["host"], ENV["precision"] = GPU, HOST, PRECISION
print(json.dumps(ENV, indent=1))

In [ ]:
# ---- model files at the pinned revision (public weights; no token) + independent revision check -------------------
REVISION_CHECK = None
MODEL_PATH = None
if GENERATOR_FACTORY is None:
    from huggingface_hub import snapshot_download, HfApi
    MODEL_PATH = snapshot_download(M["id"], revision=M["revision"], local_dir=MODEL_CACHE)
    REVISION_CHECK = C.K9.check_snapshot_revision(MODEL_PATH, M["revision"])
    C.need(HfApi().model_info(M["id"], revision=M["revision"]).sha == M["revision"], "Hugging Face did not resolve the pinned revision")
    REVISION_CHECK["hub_model_info_sha"] = M["revision"]
if COUNT_TOKENS is None:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(MODEL_PATH)
    count_tokens = lambda s: len(TOK.encode(s, add_special_tokens=False))
else:
    count_tokens = COUNT_TOKENS
print("model files:", MODEL_PATH, "| revision check:", REVISION_CHECK)

In [ ]:
# ---- fine-tuned weights: merge the LoRA adapter into the pinned base model (the generator then loads MERGED_DIR) -----
import gc, hashlib, subprocess
if GENERATOR_FACTORY is None:
    if _ver("peft") is None:
        _pins = [f"{p}=={_ver(p)}" for p in ("torch", "transformers", "accelerate") if _ver(p)]
        Path("/tmp/ft_constraints.txt").write_text("\n".join(_pins))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft", "-c", "/tmp/ft_constraints.txt"])

def _sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

if ADAPTER_DIR:
    AD = Path(ADAPTER_DIR)
else:
    _c = sorted({p.parent for p in Path(INPUT_ROOT).rglob("adapter_config.json") if "checkpoint-" not in str(p)})
    if ADAPTER_NAME:
        _c = [p for p in _c if p.name == ADAPTER_NAME]
    C.need(len(_c) == 1, f"expected exactly one adapter, found {[p.name for p in _c]}: set ADAPTER_NAME")
    AD = _c[0]
ACFG = json.loads((AD / "adapter_config.json").read_text(encoding="utf-8"))
C.need("qwen2.5-coder-3b" in str(ACFG.get("base_model_name_or_path", "")).lower(),
       f"adapter base {ACFG.get('base_model_name_or_path')} is not Qwen2.5-Coder-3B")
C.need((AD / "run_info.json").exists(), f"{AD}/run_info.json missing: can't confirm what data the adapter was trained on")
RUN_INFO = json.loads((AD / "run_info.json").read_text(encoding="utf-8"))
_used = (RUN_INFO.get("data") or {}).get("train_rows_used")
# leakage guard: the MQL-0010-safe dataset has at most 2,901 rows; the old approved_v2_1 data (3,220 train) includes
# all 716 holdout questions' databases, so an adapter trained on it must never be tested here
C.need(_used is not None and _used <= MAX_SAFE_TRAIN_ROWS,
       f"adapter {AD.name} was trained on {_used} rows (> {MAX_SAFE_TRAIN_ROWS}): it saw MQL-0010 holdout data. "
       "Use an adapter trained on approved_v2_1_mql0010_view (names end in _q10).")
C.need(AD.name.endswith("_q10"), f"adapter {AD.name} doesn't end in _q10; train on approved_v2_1_mql0010_view")
FT_ID = {"adapter_name": AD.name, "adapter_path": str(AD), "adapter_sha256": _sha_file(AD / "adapter_model.safetensors"),
         "r": ACFG.get("r"), "lora_alpha": ACFG.get("lora_alpha"), "target_modules": sorted(ACFG.get("target_modules") or []),
         "trained_on": ACFG.get("base_model_name_or_path"), "train": RUN_INFO.get("train"), "data": RUN_INFO.get("data"),
         "train_results": RUN_INFO.get("results_not_a_benchmark"),
         "applied_to": {"model": M["id"], "revision": M["revision"], "dtype": M["dtype"]},
         "method": "peft merge_and_unload into the bf16 base, saved to MERGED_DIR, loaded by the unchanged MQL-0010 generator"}

if GENERATOR_FACTORY is None:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    # Kaggle's image ships torchao 0.10; some peft versions RAISE on an old torchao instead of ignoring it. Merging a
    # plain LoRA never uses torchao, so tell every peft module it isn't available.
    for _m in list(sys.modules.values()):
        if getattr(_m, "__name__", "").startswith("peft") and hasattr(_m, "is_torchao_available"):
            _m.is_torchao_available = lambda: False
    C.need(shutil.disk_usage("/tmp").free / 2**30 >= 8, "need >= 8 GiB free in /tmp for the merged weights")
    _base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=getattr(torch, M["dtype"]))   # CPU
    _tgt = tuple(ACFG.get("target_modules") or ())
    _before = {n: p.detach().float().sum().item() for n, p in _base.named_parameters() if n.endswith(tuple(f"{t}.weight" for t in _tgt))}
    _peft = PeftModel.from_pretrained(_base, str(AD))
    _lb = [p for n, p in _peft.named_parameters() if "lora_B" in n]
    C.need(_lb and any(float(p.detach().abs().sum()) > 0 for p in _lb), "adapter has no trained LoRA weights")
    _merged = _peft.merge_and_unload()
    _after = {n: p.detach().float().sum().item() for n, p in _merged.named_parameters() if n in _before}
    _changed = sum(abs(_after[n] - _before[n]) > 0 for n in _before)
    C.need(_changed > 0, "merging the adapter changed no weight")
    _merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(MODEL_PATH).save_pretrained(MERGED_DIR)
    FT_ID["merged_weights_changed_tensors"] = f"{_changed}/{len(_before)}"
    del _base, _peft, _merged
    gc.collect()
C.write_json(Path(OUT_DIR) / "FT_ADAPTER.json", FT_ID)
print("fine-tuned adapter:", json.dumps({k: FT_ID[k] for k in ("adapter_name", "adapter_sha256", "r", "lora_alpha")}),
      "| merged tensors changed:", FT_ID.get("merged_weights_changed_tensors"))

In [ ]:
# ---- deterministic plans + prompts: re-derive A0 / D0 / E / EF and compare with the frozen copies ------------------
DOCS, PLANS, PROMPTS = C.derive_plans(P, count_tokens)
DELTA_IDS = C.delta_ids(PLANS)
C.need(DELTA_IDS == CFG["delta_ids"], "delta items differ from the frozen config")
print({a: {"budget_drops": sum(bool(r["dropped_budget"]) for r in PLANS[a]), "notes": sum(bool(r["schema_notes"]) for r in PLANS[a]),
           "max_block_tokens": max(r["context_tokens"] for r in PLANS[a])} for a in C.PLAN_ARMS}, "| delta items:", len(DELTA_IDS))

In [ ]:
# ---- preflight record -----------------------------------------------------------------------------------------------
C.need((M["id"], M["revision"]) == (C.MODEL_ID, C.MODEL_REVISION), "model identity")
C.need((M["dtype"], M["batch_size_first_pass"], M["batch_size_delta"], M["batch_size_retry"], M["max_new_tokens"], M["decoding"])
       == ("bfloat16", 8, 1, 1, 1024, "greedy (do_sample=False)"), "generation settings differ from registration A4")
PRE = {"ok": True, "at": C.now(), "session": SESSION, "passes": RUN_PASSES, "package_manifest_sha256": P["manifest_sha256"],
       "config_sha256": P["cfg_sha256"], "registration_index_sha256": CFG["registration_index_sha256"],
       "plans_equal_frozen": True, "precision": PRECISION, "revision_check": REVISION_CHECK, "env": ENV}
C.write_json(Path(OUT_DIR) / f"preflight/preflight_session{SESSION}.json", PRE)
print("PREFLIGHT OK")

In [ ]:
# ---- resume (hash-verified earlier output), then the passes with fingerprinted append-only checkpoints --------------
RESUMED = None
if RESUME_FROM:
    hits = sorted({p.parent for p in Path(INPUT_ROOT).rglob(C.STATUS_FILE)}) if RESUME_FROM == "auto" else [Path(RESUME_FROM)]
    hits = [h for h in hits if h.resolve() != Path(OUT_DIR).resolve()]
    C.need(len(hits) == 1, f"RESUME_FROM: expected exactly one earlier output, found {hits}")
    _ftp = hits[0] / "FT_ADAPTER.json"
    C.need(_ftp.exists() and json.loads(_ftp.read_text(encoding="utf-8"))["adapter_sha256"] == FT_ID["adapter_sha256"],
           f"RESUME_FROM: {hits[0]} is not a run of THIS fine-tuned adapter (base-model output or another adapter attached?)")
    _st = C.verify_resume_source(hits[0], P["manifest_sha256"])
    RESUMED = {"from": str(hits[0]), "seeded": C.seed_from_previous(hits[0], OUT_DIR), "previous_status": _st["status"]}
    print("resume source verified:", RESUMED["from"], len(RESUMED["seeded"]), "files seeded")

class SessionTimeBudget(Exception):
    pass

T_START = time.time()
OOM_EVENTS = {"batch": 0, "single_prompt": 0}
if GENERATOR_FACTORY is not None:
    RUN_BATCH, INFO, IS_OOM, CLEANUP = GENERATOR_FACTORY(CFG)
else:
    RUN_BATCH, INFO, IS_OOM, CLEANUP, _tok = C.M8.make_qwen_generator(M["id"], M["revision"], M["max_new_tokens"], M["dtype"],
                                                                        local_dir=MERGED_DIR)   # fine-tuned (merged) weights; everything else unchanged
C.need(INFO.get("dtype") == M["dtype"], f"model dtype {INFO.get('dtype')} != required {M['dtype']}")
RUNTIME = {"gpu": INFO.get("gpu"), "compute_capability": GPU.get("compute_capability"), "bf16_native": GPU.get("bf16_native"),
           "torch": INFO.get("torch"), "transformers": INFO.get("transformers"), "cuda": GPU.get("cuda")}

def _budgeted(gen):
    def g(batch):
        if time.time() - T_START > TIME_BUDGET_HOURS * 3600:
            raise SessionTimeBudget()
        return gen(batch)
    return g

GEN8 = _budgeted(C.M8.oom_safe_generator(RUN_BATCH, IS_OOM, CLEANUP, M["batch_size_first_pass"], OOM_EVENTS))
GEN1 = _budgeted(C.M8.oom_safe_generator(RUN_BATCH, IS_OOM, CLEANUP, 1, OOM_EVENTS))
STOPPED = None
for pid in RUN_PASSES:
    try:
        C.check_dependencies(OUT_DIR, P, PLANS, PROMPTS, pid)
        if pid in C.FIRST_PASSES:
            C.run_first_pass(P, pid.split(":")[1], PLANS, PROMPTS, GEN8, PRECISION, RUNTIME, OUT_DIR, STOP_AFTER_BATCHES)
        elif pid == "delta":
            C.run_delta_pass(P, PLANS, PROMPTS, GEN1, PRECISION, RUNTIME, OUT_DIR, STOP_AFTER_BATCHES)
        else:
            src = C.RETRY_PASSES[pid][1]
            C.run_retry_pass(P, pid, C.retry_sources(P, PLANS, OUT_DIR, src), PROMPTS, GEN1, PRECISION, RUNTIME, OUT_DIR,
                             STOP_AFTER_BATCHES)
    except SessionTimeBudget:
        STOPPED = f"time budget of {TIME_BUDGET_HOURS} h reached during {pid}"
        print("STOPPED:", STOPPED)
        break
    if STOP_AFTER_BATCHES is not None:
        STOPPED = "STOP_AFTER_BATCHES (test hook)"
        break

In [ ]:
# ---- export: pass completeness, environment consistency, arm assembly + verify_exports, manifest, zip -----------------
ST, MAN = C.export(P, PLANS, PROMPTS, OUT_DIR, OUTPUT_ZIP,
                   extra_status={"session": SESSION, "passes_this_session": RUN_PASSES, "resumed": RESUMED, "stopped": STOPPED,
                                 "oom_events_this_session": dict(OOM_EVENTS), "revision_check": REVISION_CHECK, "environment": ENV,
                                 "fine_tuned_adapter": FT_ID})
print("MQL-0010 STATUS:", ST["status"])
print(json.dumps(MAN["passes"], indent=1))
print("\n".join(ST["problems"]) or "no problems")
if ST["status"] != "complete":
    print("NOT COMPLETE: download", OUTPUT_ZIP, "; next version: attach this version's output, RESUME_FROM = 'auto', next SESSION.")

In [ ]:
# ---- A0-only export (FAST_A0_ONLY) -----------------------------------------------------------------------------------
# The export above needs all 8 passes, so with only first:A0 it says "partial" and writes no predictions. This builds the
# A0 arm exactly as C.assemble does (same _base_record + K9.normalize_prediction, same record format), for A0 only,
# then rewrites the checksums and the zip in the same layout as C.export.
from collections import Counter
import zipfile
if RUN_PASSES == ["first:A0"]:
    _recs = C._ck_records(C.pass_dir(OUT_DIR, "first:A0") / "checkpoint.jsonl")
    _missing = [i for i in P["ids"] if _recs.get(i, {}).get("status") != "ok"]
    C.need(not _missing, f"first:A0 incomplete ({len(_missing)} of {len(P['ids'])} missing): new version, attach this "
                         "output, RESUME_FROM = 'auto'")
    A0 = [C._base_record(_recs[i], "A0", "generated") for i in P["ids"]]
    A0 = [{**r, **C.K9.normalize_prediction(r["raw_text"], r["status"])} for r in A0]
    _pp = Path(OUT_DIR) / "predictions" / "A0" / "predictions.jsonl"
    C.write_jsonl(_pp, A0)
    C.write_json(Path(OUT_DIR) / "FT_A0_ONLY_EXPORT.json", {
        "what": "A0 arm only: fine-tuned model, MQL-0010 zero-shot prompts, built like C.assemble. Other arms not generated.",
        "n": len(A0), "predictions_sha256": C.sha256_file(_pp),
        "parse_status": dict(Counter(r["parse_status"] for r in A0)),
        "finish_reason": dict(Counter(r.get("finish_reason") for r in A0)),
        "fine_tuned_adapter": FT_ID, "written_at": C.now()})
    _od = Path(OUT_DIR)
    _names = sorted(p.relative_to(_od).as_posix() for p in _od.rglob("*")
                    if p.is_file() and p.name != "SHA256SUMS.txt" and not p.name.endswith(".tmp"))
    (_od / "SHA256SUMS.txt").write_text("".join(f"{C.sha256_file(_od / n)}  {n}\n" for n in _names), encoding="utf-8", newline="\n")
    _tmp = Path(str(OUTPUT_ZIP) + ".tmp")
    with zipfile.ZipFile(_tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for n in _names + ["SHA256SUMS.txt"]:
            z.write(_od / n, f"{_od.name}/{n}")
    os.replace(_tmp, OUTPUT_ZIP)
    print(f"A0 (fine-tuned) READY: {len(A0)} predictions | parse: {dict(Counter(r['parse_status'] for r in A0))}")
    print("download", OUTPUT_ZIP, "-> predictions/A0/predictions.jsonl is the file to score")